In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Config

In [ ]:
REPO_ROOT = "/content/drive/MyDrive/RutishauserLab/AI4Sci/METHODS/DiffusionBANMap/"
DIFFUSION_CONFIG_PATH = "config/diffusion_full_cfd.json"   # relative to repo root
DATASET_PATH = "datasets/full_cfd_val.pkl"        # relative to repo root
DIFFUSION_CKPT_PATH = None   # None → load latest checkpoint from experiment output dir
N_SAMPLES = 4
GUIDANCE_SCALE = 1.0   # >1.0 requires a CFG-trained model (cfg_uncond_prob > 0)
SEED = 42

import random, torch
random.seed(SEED)
torch.manual_seed(SEED)

## Imports

In [1]:
import os
import sys
import json
import glob
import math
import pickle

import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

sys.path.insert(0, REPO_ROOT)

import importlib.metadata
if not hasattr(importlib.metadata, '_patched'):
    _original_version = importlib.metadata.version
    def _patched_version(name):
        try:
            return _original_version(name)
        except importlib.metadata.PackageNotFoundError:
            return 'unknown'
    importlib.metadata.version = _patched_version
    importlib.metadata._patched = True

from scripts.config import BetaVAEConfig, DiffusionConfig
from models.beta_vae import BetaVAE
from models.unet import UNet
from data_utils import PreloadedDataset

## Load models

In [3]:
with open(os.path.join(REPO_ROOT, DIFFUSION_CONFIG_PATH)) as f:
    raw = json.load(f)

# Resolve paths in the config relative to REPO_ROOT
for key in ("train_dataset_path", "test_dataset_path", "output_dir",
            "betavae_config_path", "betavae_ckpt_path"):
    if key in raw and raw[key]:
        raw[key] = os.path.join(REPO_ROOT, raw[key].lstrip("./"))

diff_config = DiffusionConfig(**raw)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# --- frozen β-VAE ---
with open(diff_config.betavae_config_path) as f:
    vae_config = BetaVAEConfig(**json.load(f))

vae = BetaVAE(
    input_channels=vae_config.input_channels,
    input_height=vae_config.input_height,
    input_width=vae_config.input_width,
    latent_dim=vae_config.latent_dim,
    hidden_dim=vae_config.hidden_dim,
    beta=vae_config.beta,
    device=device,
)
vae_ckpt = torch.load(diff_config.betavae_ckpt_path, map_location=device)
vae.load_state_dict(vae_ckpt["model_state_dict"])
vae.eval()
print(f"beta-VAE loaded (step {vae_ckpt['step']})")

# --- UNet ---
unet = UNet(
    in_channels=diff_config.in_channels,
    image_size=diff_config.image_size,
    model_channels=diff_config.model_channels,
    channel_mult=diff_config.channel_mult,
    num_res_blocks=diff_config.num_res_blocks,
    attention_resolutions=diff_config.attention_resolutions,
    dropout=0.0,
    latent_dim=diff_config.latent_dim,
)

if DIFFUSION_CKPT_PATH is None:
    ckpt_dir = os.path.join(diff_config.output_dir, "checkpoints")
    ckpts = glob.glob(os.path.join(ckpt_dir, "*.pt"))
    if not ckpts:
        raise FileNotFoundError(f"No checkpoints found in {ckpt_dir}")
    DIFFUSION_CKPT_PATH = max(ckpts, key=lambda p: int(p.rsplit("step", 1)[-1].replace(".pt", "")))

diff_ckpt = torch.load(DIFFUSION_CKPT_PATH, map_location=device)
unet.load_state_dict(diff_ckpt["model_state_dict"])
unet = unet.to(device).eval()
print(f"UNet loaded: {DIFFUSION_CKPT_PATH}  (step {diff_ckpt['step']})")

β-VAE loaded (step 200)
UNet loaded: /home/jhanliufu/RutishauserLab/DiffusionBANMap/outputs/diffusion_full_cfd/checkpoints/diffusion_full_cfd_step2000.pt  (step 2000)


## Noise schedule

In [ ]:
T = diff_config.num_timesteps

def _cosine_betas(T: int, s: float = 0.008) -> torch.Tensor:
    steps = torch.arange(T + 1) / T
    f = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
    acp = f / f[0]
    return (1 - acp[1:] / acp[:-1]).clamp(max=0.999)

if diff_config.beta_schedule == "cosine":
    betas = _cosine_betas(T).to(device)
else:
    betas = torch.linspace(1e-4, 0.02, T, device=device)

alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)                              # [T]
alphas_cumprod_prev = torch.cat([torch.ones(1, device=device), alphas_cumprod[:-1]])  # [T]
sqrt_alphas_cp = alphas_cumprod.sqrt()
sqrt_one_minus_acp = (1.0 - alphas_cumprod).sqrt()
posterior_var = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)
print(f"Schedule ready: T={T}, beta_0={betas[0]:.5f}, beta_T={betas[-1]:.4f}")

Schedule ready: T=1000, beta_0=0.00004, beta_T=0.9990


## DDPM sampling

In [ ]:
@torch.no_grad()
def ddpm_sample(
    z: torch.Tensor,
    guidance_scale: float = 1.0,
    verbose: bool = False,
    n_snapshots: int = 10,
):
    """
    Generate images conditioned on beta-VAE latents z [B, latent_dim].
    guidance_scale > 1 applies classifier-free guidance (requires CFG-trained model).

    verbose=True: also returns a list of (t, image) snapshots taken at n_snapshots
    evenly-spaced timesteps during the reverse process, ordered from noisy to clean.
    """
    B = z.shape[0]
    C, H, W = diff_config.in_channels, diff_config.image_size, diff_config.image_size
    x = torch.randn(B, C, H, W, device=device)
    z_null = torch.zeros_like(z)

    # Which t_idx values to snapshot at the START of that iteration, i.e. x_t before denoising
    snapshots = None
    if verbose:
        step = max(1, T // n_snapshots)
        capture_at = set(range(T - 1, -1, -step))
        capture_at.add(0)   # always include the final step
        snapshots = []      # list of (t_idx, cpu_image_tensor)

    for t_idx in reversed(range(T)):
        if verbose and t_idx in capture_at:
            snapshots.append((t_idx, torch.sigmoid(x.clone()).cpu()))

        t_batch = torch.full((B,), t_idx, device=device, dtype=torch.long)

        if guidance_scale != 1.0:
            eps_both = unet(
                torch.cat([x, x], dim=0),
                torch.cat([t_batch, t_batch], dim=0),
                torch.cat([z, z_null], dim=0),
            )
            eps_cond, eps_uncond = eps_both.chunk(2, dim=0)
            eps_pred = eps_uncond + guidance_scale * (eps_cond - eps_uncond)
        else:
            eps_pred = unet(x, t_batch, z)

        pred_x0 = (x - sqrt_one_minus_acp[t_idx] * eps_pred) / sqrt_alphas_cp[t_idx]
        pred_x0 = pred_x0.clamp(-1.0, 1.0)

        coef1 = (alphas_cumprod_prev[t_idx].sqrt() * betas[t_idx]) / (1.0 - alphas_cumprod[t_idx])
        coef2 = (alphas[t_idx].sqrt() * (1.0 - alphas_cumprod_prev[t_idx])) / (1.0 - alphas_cumprod[t_idx])
        mean  = coef1 * pred_x0 + coef2 * x

        if t_idx > 0:
            x = mean + posterior_var[t_idx].sqrt() * torch.randn_like(x)
        else:
            x = mean

    final = torch.sigmoid(x)
    if verbose:
        # snapshots are ordered high-t → low-t (noisy → clean); append the finished x_0
        snapshots.append((0, final.cpu()))
        # deduplicate t=0 if it was already captured at the start of that iteration
        seen = set()
        snapshots = [(t, img) for t, img in snapshots if not (t in seen or seen.add(t))]

    return final, snapshots

## Sample images and generate

In [ ]:
dataset_path = os.path.join(REPO_ROOT, DATASET_PATH.lstrip("./"))
with open(dataset_path, "rb") as f:
    tensor = pickle.load(f)  # [N, C, H, W]

import random
dataset = PreloadedDataset(tensor)
indices = random.sample(range(len(dataset)), min(N_SAMPLES, len(dataset)))
originals = torch.stack([dataset[i] for i in indices]).to(device)

with torch.no_grad():
    z, _ = vae.encode(originals)           # posterior mean μ (deterministic)
    vae_recon = torch.sigmoid(vae.decode(z)).cpu()   # β-VAE reconstruction

generated, _ = ddpm_sample(z, guidance_scale=GUIDANCE_SCALE).cpu()
originals_cpu = originals.cpu()

## Visualize

In [ ]:
n = len(indices)
fig, axes = plt.subplots(3, n, figsize=(n * 2.5, 7))
fig.suptitle("original  |  β-VAE reconstruction  |  diffusion generated", fontsize=11)

rows = [originals_cpu, vae_recon, generated]
labels = ["original", "β-VAE recon", "diffusion"]

for row_idx, (imgs, label) in enumerate(zip(rows, labels)):
    axes[row_idx, 0].set_ylabel(label, fontsize=9)
    for col_idx in range(n):
        axes[row_idx, col_idx].imshow(imgs[col_idx].permute(1, 2, 0).clamp(0, 1))
        axes[row_idx, col_idx].axis("off")

plt.tight_layout()
plt.show()

## Denoising progression (verbose)

In [ ]:
# Run verbose sampling on a single image (pick the first one from the previous run)
z_single = z[:1]   # [1, latent_dim]
_, snapshots = ddpm_sample(z_single, guidance_scale=GUIDANCE_SCALE, verbose=True, n_snapshots=10)

# snapshots: list of (t_idx, [1, C, H, W]) ordered noisy → clean
n_snaps = len(snapshots)
fig, axes = plt.subplots(1, n_snaps, figsize=(n_snaps * 2, 2.8))
fig.suptitle("Reverse diffusion progression  (left: pure noise  →  right: final)", fontsize=11)

for ax, (t_idx, img) in zip(axes, snapshots):
    ax.imshow(img[0].permute(1, 2, 0).clamp(0, 1))
    ax.set_title(f"t={t_idx}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()